# PoC — Tesauro técnico multilingüe (ES ↔ EN)

**Contexto:** el corpus normativo está en **inglés** y el relato del usuario en **español**. Un tesauro curado reduce la brecha en recuperación **léxica** y apoya modelos **semánticos** débiles ante código mixto (`BL-2026-007`).

**Este notebook:** formato **solo JSON**, expansión **bidireccional**, **tags con prefijos** de dominio, modos de expansión **conservative / balanced / aggressive**, y un **flujo asistido** (LLM propone candidatos → revisión humana → fusión al tesauro oficial).

---

### Decisiones de diseño (acordadas)

| # | Tema | Decisión |
|---|------|----------|
| 1 | Dirección | **ES→EN** y **EN→ES** (retrieval, etiquetado de chunks o métricas según integres en pipeline). |
| 2 | Curación | **Asistida**: el modelo sugiere entradas; una persona valida y edita antes de mergear. |
| 3 | Ámbito | **RRS general y Team Racing / Calls**, discriminados con **prefijos** en `tags`. |
| 4 | Expansión | **Balanceada por defecto** entre conservadora y agresiva (parametrizable por llamada). |
| 5 | Variantes | **Sin** variantes regionales ES; solo equivalencias técnicas estándar. |
| 6 | Archivo | **Solo JSON** (sin YAML). Archivo canónico: `regatas_assistant/data/tesauro_tecnico_es_en.json`. |

## Formato del tesauro (JSON)

Cada entrada:

- `id`: identificador estable (`snake_case`).
- `es`: variantes en español (frases multi-palabra primero en la lista si son más seguras).
- `en`: equivalentes en inglés (orden: del más canónico en corpus a sinónimos).
- `tags`: lista con **prefijos obligatorios** para dominio, por ejemplo:
  - `rrs:*` — Reglamento / definiciones / secciones (p. ej. `rrs:section_a`, `rrs:mark`, `rrs:start`).
  - `tr:*` — Team racing / Call Book (p. ej. `tr:calls`, `tr:prestart`).
  - `proc:*` — procedimiento (p. ej. `proc:protest`, `proc:penalty`).
- `notes`: opcional; útil en curación asistida para dejar la duda abierta.

**Expansión:** `expand_for_retrieval(..., expansion="balanced")` — *conservative* toma 1 término por entrada matching; *balanced* hasta 3 (o todos si hay ≤2); *aggressive* todos.

**Uso típico:** query en ES → añadir términos EN para el retriever; texto de chunk o etiqueta en EN → añadir ES para alinear con consultas o métricas en español.

In [1]:
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

_cwd = Path.cwd().resolve()


def resolver_repo() -> Path:
    if (_cwd / "corpus").is_dir():
        return _cwd
    if (_cwd.parent / "corpus").is_dir():
        return _cwd.parent
    raise FileNotFoundError("No se encontró corpus/; ejecutá desde la raíz del repo o desde notebooks auxiliares/")


REPO = resolver_repo()
TESAURO_PATH = REPO / "regatas_assistant" / "data" / "tesauro_tecnico_es_en.json"
# Borrador del flujo asistido (revisar antes de fusionar al tesauro canónico)
TESAURO_CANDIDATOS_PATH = (
    REPO / "notebooks auxiliares" / "corpus_limpio" / "tesauro_candidatos_pendiente.json"
)

print("REPO:", REPO)
print("TESAURO_PATH:", TESAURO_PATH)
print("TESAURO_CANDIDATOS_PATH:", TESAURO_CANDIDATOS_PATH)

REPO: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final
TESAURO_PATH: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/regatas_assistant/data/tesauro_tecnico_es_en.json
TESAURO_CANDIDATOS_PATH: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/notebooks auxiliares/corpus_limpio/tesauro_candidatos_pendiente.json


In [2]:
from typing import Literal

ExpansionMode = Literal["conservative", "balanced", "aggressive"]
QueryLang = Literal["es", "en"]


def normalize_match_key(s: str) -> str:
    """Minúsculas, sin acentos, espacios colapsados (para matchear consulta y entradas)."""
    s = unicodedata.normalize("NFD", s.lower().strip())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", s)


def _phrase_pattern(normalized_phrase: str) -> re.Pattern[str]:
    parts = normalized_phrase.split()
    if not parts:
        return re.compile(r"^$")
    sep = r"\s+"
    inner = sep.join(re.escape(p) for p in parts)
    return re.compile(rf"(?<!\w){inner}(?!\w)")


def _terms_for_expansion(terms: list[str], expansion: ExpansionMode) -> list[str]:
    if not terms:
        return []
    if expansion == "conservative":
        return terms[:1]
    if expansion == "aggressive":
        return list(terms)
    if len(terms) <= 2:
        return list(terms)
    return terms[:3]


def _dedupe_terms(terms: list[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for t in terms:
        k = normalize_match_key(t)
        if k in seen:
            continue
        seen.add(k)
        out.append(t)
    return out


@dataclass
class TesauroEntry:
    id: str
    es: list[str]
    en: list[str]
    tags: list[str] | None = None
    notes: str | None = None

    def to_dict(self) -> dict:
        d = {"id": self.id, "es": self.es, "en": self.en}
        if self.tags:
            d["tags"] = self.tags
        if self.notes:
            d["notes"] = self.notes
        return d

    @staticmethod
    def from_dict(d: dict) -> TesauroEntry:
        return TesauroEntry(
            id=d["id"],
            es=list(d["es"]),
            en=list(d["en"]),
            tags=list(d["tags"]) if d.get("tags") else None,
            notes=d.get("notes"),
        )


class TesauroTecnico:
    def __init__(self, entries: list[TesauroEntry]):
        self.entries = entries
        self._patterns_es: list[tuple[TesauroEntry, str, re.Pattern[str]]] = []
        self._patterns_en: list[tuple[TesauroEntry, str, re.Pattern[str]]] = []
        for e in entries:
            for phrase in e.es:
                key = normalize_match_key(phrase)
                self._patterns_es.append((e, phrase, _phrase_pattern(key)))
            for phrase in e.en:
                key = normalize_match_key(phrase)
                self._patterns_en.append((e, phrase, _phrase_pattern(key)))
        self._patterns_es.sort(key=lambda t: len(normalize_match_key(t[1])), reverse=True)
        self._patterns_en.sort(key=lambda t: len(normalize_match_key(t[1])), reverse=True)

    def match_spanish(self, query_es: str) -> list[TesauroEntry]:
        """Entradas cuyo lado ES aparece en el texto (prioriza frases largas)."""
        return self._match_side(query_es, self._patterns_es)

    def match_english(self, query_en: str) -> list[TesauroEntry]:
        """Entradas cuyo lado EN aparece en el texto (prioriza frases largas)."""
        return self._match_side(query_en, self._patterns_en)

    def _match_side(
        self, query: str, patterns: list[tuple[TesauroEntry, str, re.Pattern[str]]]
    ) -> list[TesauroEntry]:
        q = normalize_match_key(query)
        hit: list[TesauroEntry] = []
        seen: set[str] = set()
        for entry, _phrase, rx in patterns:
            if entry.id in seen:
                continue
            if rx.search(q):
                hit.append(entry)
                seen.add(entry.id)
        return hit

    def expand_for_retrieval(
        self,
        query: str,
        *,
        source_lang: QueryLang,
        expansion: ExpansionMode = "balanced",
        separator: str = " \n",
        dedupe: bool = True,
    ) -> str:
        """Añade equivalentes en el otro idioma según matches y modo de expansión."""
        if source_lang == "es":
            hits = self.match_spanish(query)
            pool = [t for e in hits for t in _terms_for_expansion(e.en, expansion)]
        else:
            hits = self.match_english(query)
            pool = [t for e in hits for t in _terms_for_expansion(e.es, expansion)]
        if dedupe:
            pool = _dedupe_terms(pool)
        if not pool:
            return query
        return query.rstrip() + separator + separator.join(pool)

    def expand_query_for_retrieval(
        self,
        query_es: str,
        *,
        expansion: ExpansionMode = "balanced",
        separator: str = " \n",
        dedupe: bool = True,
    ) -> str:
        """Atajo ES→EN (retrieval sobre corpus EN)."""
        return self.expand_for_retrieval(
            query_es,
            source_lang="es",
            expansion=expansion,
            separator=separator,
            dedupe=dedupe,
        )

    @classmethod
    def load(cls, path: Path) -> TesauroTecnico:
        data = json.loads(path.read_text(encoding="utf-8"))
        entries = [TesauroEntry.from_dict(x) for x in data["entries"]]
        return cls(entries)

    def save(self, path: Path, *, version: str = "1") -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "schema": "regatas_tesauro_es_en",
            "version": version,
            "meta": {
                "tag_prefixes": ["rrs:", "tr:", "proc:"],
                "default_expansion": "balanced",
                "bidirectional": True,
            },
            "entries": [e.to_dict() for e in self.entries],
        }
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


## Flujo asistido (LLM → revisión humana)

Requiere **backend LLM HTTP** (`REGATAS_LLM_BACKEND=http`, URL y modelo configurados como en la app). El modelo devuelve **solo JSON** con nuevas entradas; guardalas en `TESAURO_CANDIDATOS_PATH`, revisalas y fusioná manualmente (o con celda futura de merge) al archivo canónico.

**No** ejecutes sobre corpus completo en una sola llamada: pasá **fragmentos** representativos (p. ej. un caso + definiciones relevantes).


In [3]:
def _strip_json_fence(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        lines = t.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        t = "\n".join(lines)
    return t.strip()


def proponer_entradas_con_llm(
    contexto_corpus: str,
    *,
    consulta_es_ejemplo: str | None = None,
    max_entradas: int = 15,
    model_override: str | None = None,
) -> dict:
    """Devuelve {\"entries\": [...]} para grabar en TESAURO_CANDIDATOS_PATH."""
    from openai import OpenAI

    from regatas_assistant.config import Settings

    s = Settings.from_env()
    if s.llm_backend == "stub":
        raise RuntimeError(
            "REGATAS_LLM_BACKEND=stub no puede llamar al LLM. Configurá HTTP (p. ej. Ollama)."
        )
    base = s.llm_base_url or "http://127.0.0.1:11434/v1"
    key = s.llm_api_key or "ollama"
    model = model_override or s.llm_model

    user_extra = ""
    if consulta_es_ejemplo:
        user_extra = (
            "\n\nConsulta de usuario (ES) para orientar términos:\n" + consulta_es_ejemplo
        )

    system = (
        "Eres experto en regatas (RRS y Team Racing). Devuelves SOLO JSON válido (sin markdown). "
        'El JSON debe tener la forma: {"entries": [ {"id": "snake_case", "es": [...], "en": [...], '
        '"tags": ["rrs:...", "tr:...", "proc:..."], "notes": "opcional" }, ... ]}. '
        "Reglas: ids únicos en el lote; cada tag con prefijo rrs:, tr:, o proc:; "
        "equivalencias técnicas estándar (sin variantes regionales ES)."
    )

    user = (
        f"Fragmentos del corpus normativo (EN):\n\n{contexto_corpus[:12000]}"
        f"{user_extra}\n\n"
        f"Propón hasta {max_entradas} entradas NUEVAS útiles para alinear consultas ES con texto EN."
    )

    client = OpenAI(base_url=base, api_key=key)
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content or ""
    data = json.loads(_strip_json_fence(raw))
    if "entries" not in data:
        raise ValueError("La respuesta del modelo no contiene la clave 'entries'.")
    return data


# Ejemplo (descomentar; requiere LLM HTTP y fragmento real del corpus):
# limpio = REPO / "notebooks auxiliares" / "corpus_limpio" / "WS-Case-Book-2025-2028-v2025-07_limpio.txt"
# ctx = limpio.read_text(encoding="utf-8")[:8000]
# borrador = proponer_entradas_con_llm(ctx, consulta_es_ejemplo="¿Quién debía mantenerse alejado cerca de la boya?")
# TESAURO_CANDIDATOS_PATH.parent.mkdir(parents=True, exist_ok=True)
# TESAURO_CANDIDATOS_PATH.write_text(json.dumps(borrador, ensure_ascii=False, indent=2), encoding="utf-8")
# print("Revisar:", TESAURO_CANDIDATOS_PATH.resolve())


## Conjunto inicial (curado mínimo)

Ampliá esta lista desde el Case Book / RRS / Calls. Los `id` no deben duplicarse. Los `tags` usan prefijos `rrs:`, `tr:`, `proc:`.



In [4]:
ENTRADAS_INICIALES: list[TesauroEntry] = [
    TesauroEntry(
        id="keep_clear",
        es=["mantenerse alejado", "mantenerse alejada", "mantenerse alejadas", "mantenerse alejados"],
        en=["keep clear", "keeping clear", "keeps clear"],
        tags=["rrs:section_a", "rrs:right_of_way"],
        notes="RRS: obligación del barco sin derecho de paso.",
    ),
    TesauroEntry(
        id="right_of_way",
        es=["derecho de paso", "barco con derecho de paso"],
        en=["right of way", "right-of-way", "ROW"],
        tags=["rrs:section_a"],
    ),
    TesauroEntry(
        id="give_way",
        es=["ceder el paso"],
        en=["give way", "giving way"],
        tags=["rrs:section_a"],
    ),
    TesauroEntry(
        id="room",
        es=["espacio", "espacio suficiente", "dar espacio"],
        en=["room", "give room", "giving room"],
        tags=["rrs:section_b", "rrs:mark"],
        notes="Cuidado: 'room' también en Rule 18 mark-room.",
    ),
    TesauroEntry(
        id="mark_room",
        es=["margen de boya", "regla 18", "zona de boya"],
        en=["mark-room", "Rule 18", "zone of a mark"],
        tags=["rrs:mark", "rrs:rule18", "tr:calls"],
    ),
    TesauroEntry(
        id="tack_tacking",
        es=["virar", "virando", "virada", "cambio de amura"],
        en=["tack", "tacking"],
        tags=["rrs:maneuver"],
    ),
    TesauroEntry(
        id="gybe_jibe",
        es=["empopar", "empopando", "empopada"],
        en=["gybe", "gybing", "jibe", "jibing"],
        tags=["rrs:maneuver"],
    ),
    TesauroEntry(
        id="port_starboard",
        es=["babor", "estribor", "barco de babor", "barco de estribor"],
        en=["port", "starboard", "port-tack", "starboard-tack"],
        tags=["rrs:section_a", "rrs:tack"],
    ),
    TesauroEntry(
        id="windward_leeward",
        es=["barlovento", "sotavento", "a barlovento", "a sotavento"],
        en=["windward", "leeward"],
        tags=["rrs:section_a"],
    ),
    TesauroEntry(
        id="overlap",
        es=["solapamiento", "solapados", "solapadas"],
        en=["overlap", "overlapped", "clear astern", "clear ahead"],
        tags=["rrs:definitions"],
        notes="Revisar falsos positivos en queries muy cortas.",
    ),
    TesauroEntry(
        id="obstruction",
        es=["obstrucción", "obstruir"],
        en=["obstruction", "obstructing"],
        tags=["rrs:definitions"],
    ),
    TesauroEntry(
        id="proper_course",
        es=["rumbo propio", "rumbo apropiado"],
        en=["proper course"],
        tags=["rrs:rule17"],
    ),
    TesauroEntry(
        id="fetch_mark",
        es=["pasar la boya", "bordear la boya", "en la boya"],
        en=["fetching the mark", "passing the mark", "round the mark"],
        tags=["rrs:mark"],
    ),
    TesauroEntry(
        id="protest",
        es=["protesta", "protestar", "protestó"],
        en=["protest", "protesting"],
        tags=["proc:protest"],
    ),
    TesauroEntry(
        id="penalty",
        es=["penalización", "penalidad", "vueltas de penalización", "giros de penalización"],
        en=["penalty", "penalty turns", "take a penalty"],
        tags=["proc:penalty"],
    ),
    TesauroEntry(
        id="start_line",
        es=["línea de salida", "pre salida", "previo a la salida"],
        en=["starting line", "pre-start", "before the start"],
        tags=["rrs:start", "tr:prestart"],
    ),
    TesauroEntry(
        id="tr_team_racing_turn",
        es=["viraje en team racing", "giro de equipo"],
        en=["team racing turn"],
        tags=["tr:team_racing", "tr:calls"],
        notes="Verificar contra Call Book TR; ajustar ES si tu circle usa otro término.",
    ),
]

tesauro = TesauroTecnico(ENTRADAS_INICIALES)
tesauro.save(TESAURO_PATH, version="1")
print("Guardado:", TESAURO_PATH.resolve())
print("Entradas:", len(tesauro.entries))


Guardado: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/regatas_assistant/data/tesauro_tecnico_es_en.json
Entradas: 17


## Demostración: expansión bidireccional

Probá los tres modos en ES→EN y un ejemplo EN→ES (útil si indexás metadatos o queries mixtas).



In [5]:
ejemplos_es = [
    "El barco de babor no se mantuvo alejado al virar cerca de la boya.",
    "¿Quién tenía derecho de paso en la zona de boya con solapamiento?",
]

for q in ejemplos_es:
    print("---")
    print("ES:", q)
    for mode in ("conservative", "balanced", "aggressive"):
        exp = tesauro.expand_for_retrieval(q, source_lang="es", expansion=mode)
        print(f"  [{mode}]", exp.replace(" \n", " | "))
    print("  IDs:", [h.id for h in tesauro.match_spanish(q)])

ejemplos_en = [
    "Did the starboard-tack boat keep clear while fetching the mark?",
]

for q in ejemplos_en:
    print("---")
    print("EN:", q)
    exp = tesauro.expand_for_retrieval(q, source_lang="en", expansion="balanced")
    print("  [balanced ES inject]", exp.replace(" \n", " | "))
    print("  IDs:", [h.id for h in tesauro.match_english(q)])


---
ES: El barco de babor no se mantuvo alejado al virar cerca de la boya.
  [conservative] El barco de babor no se mantuvo alejado al virar cerca de la boya. | port | tack
  [balanced] El barco de babor no se mantuvo alejado al virar cerca de la boya. | port | starboard | port-tack | tack | tacking
  [aggressive] El barco de babor no se mantuvo alejado al virar cerca de la boya. | port | starboard | port-tack | starboard-tack | tack | tacking
  IDs: ['port_starboard', 'tack_tacking']
---
ES: ¿Quién tenía derecho de paso en la zona de boya con solapamiento?
  [conservative] ¿Quién tenía derecho de paso en la zona de boya con solapamiento? | right of way | mark-room | overlap
  [balanced] ¿Quién tenía derecho de paso en la zona de boya con solapamiento? | right of way | right-of-way | ROW | mark-room | Rule 18 | zone of a mark | overlap | overlapped | clear astern
  [aggressive] ¿Quién tenía derecho de paso en la zona de boya con solapamiento? | right of way | right-of-way | ROW | mark-

## Cargar desde disco (como lo haría el pipeline)

`TesauroTecnico.load(TESAURO_PATH)` para no depender de haber ejecutado la celda del seed en esta sesión.



In [6]:
tesauro_disk = TesauroTecnico.load(TESAURO_PATH)
_ne = len(json.loads(TESAURO_PATH.read_text(encoding="utf-8"))["entries"])
assert len(tesauro_disk.entries) == _ne
print("OK load:", TESAURO_PATH)


OK load: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/regatas_assistant/data/tesauro_tecnico_es_en.json


---

### Próximos pasos sugeridos

- Extraer candidatos desde `corpus_limpio/*_limpio.txt` y pasarlos en **trozos** al flujo asistido.
- Medir **recall@k** con `expand_for_retrieval(..., expansion="balanced")` vs conservative/aggressive.
- Integrar en `CorpusRetriever.retrieve`: query ES del usuario → expansión ES→EN; opcionalmente EN→ES sobre texto de chunk para etiquetas (`BL-2026-007`).
- Tras revisar `tesauro_candidatos_pendiente.json`, fusionar entradas válidas al JSON canónico y subir `version` en `save`.

